# Deepfix数据集处理

## 1 依赖导入与全局配置

In [1]:
import sqlite3
import json
import os

# =========================================================
# 全局配置：统一管理所有可调参数
# =========================================================
DB_PATH      = os.path.join('data', 'database', 'prutor-deepfix.db')
OUTPUT_FILE  = 'deepfix.json'
MIN_CODE_LEN = 35  # 代码最短字符数（过滤过短的无意义代码）
MAX_CODE_LEN = 65535   # 代码最长字符数（过滤过长的异常代码）

# 数据库列名配置（根据实际列名调整）
ID_COL       = 'code_id'
USER_ID_COL  = 'user_id'
PROBLEM_ID_COL = 'problem_id'
CODE_COL     = 'code'
ERROR_COL    = 'error'
ERRORCNT_COL = 'errorcount'

print("✅ 配置加载完成")
print(f"   数据库路径: {DB_PATH}")
print(f"   输出文件:   {OUTPUT_FILE}")
print(f"   最短代码长度: {MIN_CODE_LEN}")


✅ 配置加载完成
   数据库路径: data\database\prutor-deepfix.db
   输出文件:   deepfix.json
   最短代码长度: 35


## 2 工具函数

### 数据库连接与结构探查

In [2]:
def get_connection(db_path: str) -> sqlite3.Connection:
    """建立并返回数据库连接"""
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"找不到数据库文件: {db_path}")
    conn = sqlite3.connect(db_path)
    print(f"✅ 数据库连接成功: {db_path}")
    return conn


def inspect_table_schema(conn: sqlite3.Connection, table_name: str = 'Code') -> list:
    """
    探查指定表的列结构，返回列名列表
    可用于在正式查询前确认列名是否与配置一致
    """
    cursor = conn.cursor()
    cursor.execute(f"PRAGMA table_info({table_name});")
    columns = cursor.fetchall()

    print(f"\n📋 表 [{table_name}] 的列结构：")
    print(f"  {'cid':<5} {'name':<20} {'type':<15} {'notnull':<10} {'default':<15} {'pk'}")
    print(f"  {'-'*70}")
    for col in columns:
        print(f"  {col[0]:<5} {col[1]:<20} {col[2]:<15} {col[3]:<10} {str(col[4]):<15} {col[5]}")

    col_names = [col[1] for col in columns]
    return col_names


# ---- 运行探查 ----
conn = get_connection(DB_PATH)
col_names = inspect_table_schema(conn, 'Code')

✅ 数据库连接成功: data\database\prutor-deepfix.db

📋 表 [Code] 的列结构：
  cid   name                 type            notnull    default         pk
  ----------------------------------------------------------------------
  0     code_id              TEXT            0          None            1
  1     user_id              TEXT            0          None            0
  2     problem_id           TEXT            0          None            0
  3     code                 TEXT            0          None            0
  4     error                TEXT            0          None            0
  5     errorcount           INT             0          None            0


### 查询所有有错误的代码记录

In [3]:
def fetch_buggy_records(
    conn: sqlite3.Connection,
    code_col: str,
    error_col: str,
    errorcnt_col: str,
    min_code_len: int
) -> list:
    """
    从数据库中查询所有 errorcount > 0 的 buggy 代码记录
    返回: List of (code_id, user_id, problem_id, code, error_msg)
    """
    query = f"""
        SELECT code_id, user_id, problem_id, {code_col}, {error_col}
        FROM Code
        WHERE {errorcnt_col} > 0
          AND LENGTH({code_col}) >= {min_code_len}
    """
    cursor = conn.cursor()
    cursor.execute(query)
    rows = cursor.fetchall()

    print(f"✅ 查询完成：共找到 {len(rows)} 条有错误的代码记录")
    return rows


# ---- 运行查询 ----
buggy_records = fetch_buggy_records(
    conn, CODE_COL, ERROR_COL, ERRORCNT_COL, MIN_CODE_LEN
)
print(f"buggy记录数量：{len(buggy_records)}")

# 预览前 3 条
print("\n🔍 预览前 3 条 buggy 记录：")
for record in buggy_records[:3]:
    code_id, user_id, problem_id, code, error_msg = record
    print(f"  code_id={code_id}, user_id={user_id}, "
          f"problem_id={problem_id}, errorMsg={str(error_msg)[:40]}...")


✅ 查询完成：共找到 6978 条有错误的代码记录
buggy记录数量：6978

🔍 预览前 3 条 buggy 记录：
  code_id=prog19941, user_id=user801, problem_id=prob90, errorMsg=In function ‘main’:
18:30: error: expect...
  code_id=prog00278, user_id=user481, problem_id=prob90, errorMsg=In function ‘main’:
8:9: error: ‘i’ unde...
  code_id=prog31047, user_id=user507, problem_id=prob90, errorMsg=In function ‘main’:
16:5: error: expecte...


### 为单条 buggy 代码查找对应的正确代码

In [4]:
def fetch_fixed_code(
    cursor: sqlite3.Cursor,
    user_id: str,
    problem_id: str,
    code_col: str,
    errorcnt_col: str,
    min_code_len: int
) -> str | None:
    """
    为指定 (user_id, problem_id) 查找该用户对同一题目【最终提交】的正确代码。

    修改说明：
        原版本使用 LIMIT 1 不加排序，会随机返回任意一条正确记录；
        由于扫描结果显示 81.3%（6910/8497）的 (user, problem) 对
        存在多次正确提交，且绝大多数版本各不相同（6905/6910），
        随机取值会引入大量噪声——同一道题的早期正确版本可能仍
        存在逻辑缺陷或风格问题，不能代表用户最终的解题意图。
        因此改为按 code_id 降序排列，取 code_id 最大的一条，
        以此作为用户"最终提交"的正确版本。

    选取"最终提交"的依据：
        code_id 在 Prutor 系统中为自增式编号（如 prog00001, prog59999），
        字典序降序等价于提交时间降序，code_id 最大者即为最晚提交的版本，
        最能代表用户经过多次迭代后趋于稳定的最终解答。

    参数:
        cursor       : 数据库游标（由调用方统一管理）
        user_id      : 学生 ID
        problem_id   : 题目 ID
        code_col     : 代码列名
        errorcnt_col : 错误数量列名
        min_code_len : 过滤代码的最小字符数

    返回:
        最终正确提交的代码字符串；若该用户从未正确提交过则返回 None
    """
    query = f"""
        SELECT {code_col}
        FROM Code
        WHERE user_id      = ?
          AND problem_id   = ?
          AND {errorcnt_col} = 0
          AND LENGTH({code_col}) >= ?
        ORDER BY code_id DESC   -- ← 核心修改：按 code_id 降序，取最晚提交的版本
        LIMIT 1
    """
    cursor.execute(query, (user_id, problem_id, min_code_len))
    row = cursor.fetchone()
    return row[0] if row is not None else None


def is_valid_pair(buggy_code: str, fixed_code: str) -> bool:
    """
    校验 (buggy, fix) 对是否有效：
    过滤 buggy 与 fix 内容完全相同的极端情况
    """
    return buggy_code.strip() != fixed_code.strip()


# ================================================================
# 单条测试：对比修改前后的差异，验证函数是否正常工作
# ================================================================
if buggy_records:
    test_id, test_user, test_prob, test_code, test_err = buggy_records[0]
    test_cursor = conn.cursor()

    # ── 修改后：取最终版本 ──────────────────────────────────────────
    test_fix_latest = fetch_fixed_code(
        test_cursor, test_user, test_prob,
        CODE_COL, ERRORCNT_COL, MIN_CODE_LEN
    )

    # ── 修改前（仅用于对比）：不加排序直接 LIMIT 1 ──────────────────
    test_cursor.execute(f"""
        SELECT {CODE_COL} FROM Code
        WHERE user_id = ? AND problem_id = ?
          AND {ERRORCNT_COL} = 0 AND LENGTH({CODE_COL}) >= ?
        LIMIT 1
    """, (test_user, test_prob, MIN_CODE_LEN))
    row_old = test_cursor.fetchone()
    test_fix_old = row_old[0] if row_old else None

    print(f"🔍 测试配对：user={test_user}, problem={test_prob}")
    print(f"   找到正确代码        : {'✅ 是' if test_fix_latest else '❌ 否（该用户未提交过正确版本）'}")

    if test_fix_latest:
        print(f"   有效配对            : {'✅ 是' if is_valid_pair(test_code, test_fix_latest) else '❌ 否（内容完全相同）'}")
        print()

        # 对比新旧版本是否相同（若用户只有一次正确提交则两者相同）
        versions_differ = (test_fix_old != test_fix_latest)
        print(f"   ⚠️  修改前后取到的正确代码是否不同: {'✅ 是（多版本场景，修改有效）' if versions_differ else '➖ 否（该用户只有单次正确提交）'}")
        print()
        print(f"   【buggy 代码（前5行）】")
        for line in test_code.strip().split('\n')[:5]:
            print(f"     {line}")
        print()
        print(f"   【修改前取到的正确代码（前5行）】")
        if test_fix_old:
            for line in test_fix_old.strip().split('\n')[:5]:
                print(f"     {line}")
        print()
        print(f"   【修改后取到的最终正确代码（前5行）】")
        for line in test_fix_latest.strip().split('\n')[:5]:
            print(f"     {line}")


🔍 测试配对：user=user801, problem=prob90
   找到正确代码        : ✅ 是
   有效配对            : ✅ 是

   ⚠️  修改前后取到的正确代码是否不同: ➖ 否（该用户只有单次正确提交）

   【buggy 代码（前5行）】
     #include <stdio.h>
     int main(){
         int k,n;
         scanf("%d%d",&k,&n);
         int a[n];

   【修改前取到的正确代码（前5行）】
     #include <stdio.h>
     int main(){
         int k,n;
         scanf("%d%d",&k,&n);
         int a[n];

   【修改后取到的最终正确代码（前5行）】
     #include <stdio.h>
     int main(){
         int k,n;
         scanf("%d%d",&k,&n);
         int a[n];


### 构建单条数据集条目

In [5]:
def build_entry(
    code_id: str,
    user_id: str,
    problem_id: str,
    buggy_code: str,
    fixed_code: str,
    error_msg: str
) -> dict:
    """
    将配对成功的 (buggy, fix) 封装为标准 JSON 条目格式
    """
    return {
        "id"           : str(code_id),
        "user_id"      : str(user_id),
        "problem_id"   : str(problem_id),
        "bug_type"     : "common_programming_error",
        "language"     : "C",
        "buggy_code"   : str(buggy_code).strip(),
        "fixed_code"   : str(fixed_code).strip(),
        "error_message": str(error_msg).strip() if error_msg else ""
    }


# ---- 单条测试：预览条目结构 ----
if buggy_records and test_fix_latest:
    sample_entry = build_entry(
        test_id, test_user, test_prob,
        test_code, test_fix_latest, test_err
    )
    print("🔍 示例条目结构：")
    for k, v in sample_entry.items():
        preview = str(v)[:60].replace('\n', '\\n')
        print(f"  {k:<15}: {preview}{'...' if len(str(v)) > 60 else ''}")


🔍 示例条目结构：
  id             : prog19941
  user_id        : user801
  problem_id     : prob90
  bug_type       : common_programming_error
  language       : C
  buggy_code     : #include <stdio.h>\nint main(){\n    int k,n;\n    scanf("%d%d"...
  fixed_code     : #include <stdio.h>\nint main(){\n    int k,n;\n    scanf("%d%d"...
  error_message  : In function ‘main’:\n18:30: error: expected expression before...


### 判断用户针对某个问题是否有多次正确的提交

In [6]:
import sqlite3
import os
from collections import defaultdict


# ================================================================
# 工具函数 1（原有）：检查单个用户对单个题目的多次正确提交情况
# ================================================================
def check_multiple_correct_submissions(
    db_path: str,
    user_id: str,
    problem_id: str,
    errorcnt_col: str = 'errorcount',
    code_col: str = 'code',
    min_code_len: int = 35,
) -> dict:
    """
    判断数据库中某个用户针对某个问题是否存在多次正确提交。

    本函数完全上下文无关，自行管理数据库连接的建立与释放，
    可在 Notebook 的任意 Cell 中独立调用。

    参数:
        db_path      : 数据库文件的路径
        user_id      : 要查询的学生 ID（如 'user696'）
        problem_id   : 要查询的题目 ID（如 'prob61'）
        errorcnt_col : 错误数量列名，默认 'errorcount'
        code_col     : 代码列名，默认 'code'
        min_code_len : 过滤代码的最小字符数，默认 35

    返回:
        {
            'user_id'         : str,   # 查询的用户 ID
            'problem_id'      : str,   # 查询的题目 ID
            'has_multiple'    : bool,  # 是否存在多次正确提交
            'total_count'     : int,   # 正确提交的总次数
            'unique_count'    : int,   # 去重后不同代码版本的数量
            'is_all_identical': bool,  # 所有正确提交的代码内容是否完全一致
            'code_ids'        : list,  # 所有正确提交的 code_id 列表
        }

    异常:
        FileNotFoundError : 数据库文件不存在时抛出
        RuntimeError      : 数据库查询失败时抛出
    """
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"数据库文件不存在: {db_path}")

    query = f"""
        SELECT code_id, {code_col}
        FROM Code
        WHERE user_id      = ?
          AND problem_id   = ?
          AND {errorcnt_col} = 0
          AND LENGTH({code_col}) >= ?
        ORDER BY code_id
    """

    try:
        with sqlite3.connect(db_path) as conn:
            cursor = conn.cursor()
            cursor.execute(query, (user_id, problem_id, min_code_len))
            rows = cursor.fetchall()
    except sqlite3.Error as e:
        raise RuntimeError(f"数据库查询失败: {e}")

    code_ids     = [row[0] for row in rows]
    unique_codes = set(row[1].strip() for row in rows)
    total_count  = len(rows)
    unique_count = len(unique_codes)

    return {
        'user_id'         : user_id,
        'problem_id'      : problem_id,
        'has_multiple'    : total_count > 1,
        'total_count'     : total_count,
        'unique_count'    : unique_count,
        'is_all_identical': unique_count <= 1,
        'code_ids'        : code_ids,
    }


# ================================================================
# 工具函数 2（新增）：扫描全表，统计所有用户的多次正确提交情况
# ================================================================
def scan_all_multiple_correct_submissions(
    db_path: str,
    errorcnt_col: str = 'errorcount',
    code_col: str = 'code',
    min_code_len: int = 35,
) -> dict:
    """
    扫描数据库中所有用户的全部正确提交记录，
    统计哪些 (user_id, problem_id) 组合存在多次正确提交行为。

    本函数一次性拉取全表的正确提交记录并在内存中完成聚合，
    相比对每个用户单独查询，效率大幅提升。
    完全上下文无关，可在 Notebook 任意 Cell 中独立调用。

    参数:
        db_path      : 数据库文件的路径
        errorcnt_col : 错误数量列名，默认 'errorcount'
        code_col     : 代码列名，默认 'code'
        min_code_len : 过滤代码的最小字符数，默认 35

    返回:
        {
            'total_correct_records' : int,   # 满足条件的正确提交总记录数
            'total_unique_pairs'    : int,   # 不同的 (user_id, problem_id) 对总数
            'pairs_with_multiple'   : int,   # 存在多次正确提交的对数
            'pairs_all_identical'   : int,   # 多次提交但代码完全相同的对数
            'pairs_with_diff_code'  : int,   # 多次提交且代码存在差异的对数
            'details'               : [      # 仅包含多次提交的条目列表，按提交次数降序排列
                {
                    'user_id'         : str,
                    'problem_id'      : str,
                    'total_count'     : int,
                    'unique_count'    : int,
                    'is_all_identical': bool,
                    'code_ids'        : list,
                },
                ...
            ]
        }

    异常:
        FileNotFoundError : 数据库文件不存在时抛出
        RuntimeError      : 数据库查询失败时抛出
    """
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"数据库文件不存在: {db_path}")

    # ── 第一步：一次性拉取所有正确提交记录 ──────────────────────────
    query = f"""
        SELECT user_id, problem_id, code_id, {code_col}
        FROM Code
        WHERE {errorcnt_col} = 0
          AND LENGTH({code_col}) >= ?
        ORDER BY user_id, problem_id, code_id
    """

    try:
        with sqlite3.connect(db_path) as conn:
            cursor = conn.cursor()
            cursor.execute(query, (min_code_len,))
            rows = cursor.fetchall()
    except sqlite3.Error as e:
        raise RuntimeError(f"数据库查询失败: {e}")

    # ── 第二步：在内存中按 (user_id, problem_id) 聚合 ───────────────
    # 结构: { (user_id, problem_id): {'code_ids': [], 'code_set': set()} }
    pair_map = defaultdict(lambda: {'code_ids': [], 'code_set': set()})

    for user_id, problem_id, code_id, code_content in rows:
        key = (str(user_id), str(problem_id))
        pair_map[key]['code_ids'].append(str(code_id))
        pair_map[key]['code_set'].add(code_content.strip())

    # ── 第三步：统计汇总 ─────────────────────────────────────────────
    details            = []
    pairs_with_multiple = 0
    pairs_all_identical = 0
    pairs_with_diff     = 0

    for (user_id, problem_id), data in pair_map.items():
        total_count  = len(data['code_ids'])
        unique_count = len(data['code_set'])

        if total_count <= 1:
            # 只有一次正确提交，不纳入 details
            continue

        # 存在多次正确提交
        pairs_with_multiple += 1
        is_identical = (unique_count == 1)

        if is_identical:
            pairs_all_identical += 1
        else:
            pairs_with_diff += 1

        details.append({
            'user_id'         : user_id,
            'problem_id'      : problem_id,
            'total_count'     : total_count,
            'unique_count'    : unique_count,
            'is_all_identical': is_identical,
            'code_ids'        : data['code_ids'],
        })

    # 按提交次数降序排列，方便查看最频繁的用户
    details.sort(key=lambda x: x['total_count'], reverse=True)

    return {
        'total_correct_records': len(rows),
        'total_unique_pairs'   : len(pair_map),
        'pairs_with_multiple'  : pairs_with_multiple,
        'pairs_all_identical'  : pairs_all_identical,
        'pairs_with_diff_code' : pairs_with_diff,
        'details'              : details,
    }


# ================================================================
# 快速验证入口
# ================================================================
if __name__ == '__main__' or True:
    _DB_PATH = os.path.join('data', 'database', 'prutor-deepfix.db')

    # ── 验证 1：单用户单题目查询（原有功能）────────────────────────
    print("【单用户查询示例】")
    result = check_multiple_correct_submissions(_DB_PATH, 'user696', 'prob61')
    print("=" * 45)
    print(f"  用户 ID        : {result['user_id']}")
    print(f"  题目 ID        : {result['problem_id']}")
    print(f"  是否多次正确提交: {'✅ 是' if result['has_multiple'] else '❌ 否'}")
    print(f"  正确提交总次数  : {result['total_count']}")
    print(f"  不同代码版本数  : {result['unique_count']}")
    print(f"  所有版本完全相同: {'✅ 是' if result['is_all_identical'] else '❌ 否（存在不同版本）'}")
    print(f"  提交记录 ID 列表: {result['code_ids'] if result['code_ids'] else '无正确提交记录'}")
    print("=" * 45)

    # ── 验证 2：全表扫描（新增功能）────────────────────────────────
    print("\n【全表扫描结果】")
    scan = scan_all_multiple_correct_submissions(_DB_PATH)
    print("=" * 45)
    print(f"  正确提交总记录数        : {scan['total_correct_records']}")
    print(f"  (用户, 题目) 唯一对总数  : {scan['total_unique_pairs']}")
    print(f"  存在多次正确提交的对数   : {scan['pairs_with_multiple']}")
    print(f"  ├─ 多次提交且完全相同    : {scan['pairs_all_identical']}")
    print(f"  └─ 多次提交但存在差异    : {scan['pairs_with_diff_code']}")
    print("=" * 45)

    print(f"\n  📋 提交次数最多的前 5 个 (用户, 题目) 对：")
    for entry in scan['details'][:5]:
        identical_tag = '（内容完全相同）' if entry['is_all_identical'] else '（存在不同版本）'
        print(f"  user={entry['user_id']}, problem={entry['problem_id']} "
              f"| 共 {entry['total_count']} 次, {entry['unique_count']} 个不同版本 {identical_tag}")
        print(f"    code_ids: {entry['code_ids']}")


【单用户查询示例】
  用户 ID        : user696
  题目 ID        : prob61
  是否多次正确提交: ✅ 是
  正确提交总次数  : 12
  不同代码版本数  : 12
  所有版本完全相同: ❌ 否（存在不同版本）
  提交记录 ID 列表: ['prog04647', 'prog12101', 'prog17917', 'prog23676', 'prog25551', 'prog28052', 'prog48094', 'prog49937', 'prog52286', 'prog53904', 'prog56735', 'prog59094']

【全表扫描结果】
  正确提交总记录数        : 46500
  (用户, 题目) 唯一对总数  : 8497
  存在多次正确提交的对数   : 6910
  ├─ 多次提交且完全相同    : 5
  └─ 多次提交但存在差异    : 6905

  📋 提交次数最多的前 5 个 (用户, 题目) 对：
  user=user493, problem=prob288 | 共 135 次, 135 个不同版本 （存在不同版本）
    code_ids: ['prog00298', 'prog02118', 'prog03237', 'prog03364', 'prog03515', 'prog03885', 'prog05009', 'prog05314', 'prog06162', 'prog07281', 'prog07927', 'prog08078', 'prog08175', 'prog08493', 'prog08755', 'prog08948', 'prog09168', 'prog09507', 'prog09939', 'prog10016', 'prog10645', 'prog11119', 'prog11214', 'prog11945', 'prog12653', 'prog13756', 'prog13905', 'prog14083', 'prog14349', 'prog15024', 'prog17217', 'prog18446', 'prog20491', 'prog20732', 'prog20786', 'prog

## 3 数据集生成

In [9]:
def extract_real_bug_fix_pairs(
    conn: sqlite3.Connection,
    buggy_records: list,
    code_col: str,
    errorcnt_col: str,
    max_code_len: int = 65535,  
    min_code_len: int = 35,
    log_interval: int = 500
) -> tuple[list, dict]:
    """
    核心配对流程：遍历所有 buggy 记录，为每条记录寻找对应的正确版本
    返回: (dataset列表, 统计信息字典)
    """
    cursor   = conn.cursor()
    dataset  = []
    skipped_no_fix   = 0   # 该用户从未提交过正确版本
    skipped_identical = 0  # buggy 与 fix 内容完全相同
    skipped_too_long  = 0   

    for row in buggy_records:
        code_id, user_id, problem_id, buggy_code, error_msg = row

        # Step 1: 查找对应正确代码
        fixed_code = fetch_fixed_code(
            cursor, user_id, problem_id,
            code_col, errorcnt_col, min_code_len
        )
        if fixed_code is None:
            skipped_no_fix += 1
            continue
        
        # Step 2: 过滤过长代码（buggy 或 fixed 任一超限即丢弃）← 新增  
        if len(buggy_code) > max_code_len or len(fixed_code) > max_code_len:  
            skipped_too_long += 1  
            continue  

        # Step 3: 过滤无效配对
        if not is_valid_pair(buggy_code, fixed_code):
            skipped_identical += 1
            continue

        # Step 4: 构建条目并写入
        entry = build_entry(
            code_id, user_id, problem_id,
            buggy_code, fixed_code, error_msg
        )
        dataset.append(entry)

        if len(dataset) % log_interval == 0:
            print(f"  ⏳ 已生成 {len(dataset)} 条数据...")

    stats = {
        "total_buggy"       : len(buggy_records),
        "successfully_paired": len(dataset),
        "skipped_no_fix"    : skipped_no_fix,
        "skipped_identical" : skipped_identical,
        "skipped_too_long"   : skipped_too_long,
    }
    return dataset, stats

def save_dataset(dataset: list, output_file: str) -> None:
    """将数据集保存为 UTF-8 编码的 JSON 文件"""
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(dataset, f, indent=2, ensure_ascii=False)
    size_kb = os.path.getsize(output_file) / 1024
    print(f"✅ 数据集已保存至: {output_file}  ({size_kb:.1f} KB)")


# ---- 运行核心配对流程 ----
print("🚀 开始批量配对...\n")
dataset, stats = extract_real_bug_fix_pairs(
    conn, buggy_records,
    CODE_COL, ERRORCNT_COL
)
print("\n========== 配对完成 ==========")
print(f"  总 buggy 记录数    : {stats['total_buggy']}")
print(f"  成功配对           : {stats['successfully_paired']}")
print(f"  跳过(无正确版本)   : {stats['skipped_no_fix']}")
print(f"  跳过(内容完全相同) : {stats['skipped_identical']}")
print(f"  跳过(代码过长)     : {stats['skipped_too_long']}")   

# ---- 保存 & 关闭 ----
save_dataset(dataset, OUTPUT_FILE)
conn.close()
print("✅ 数据库连接已关闭")



🚀 开始批量配对...

  ⏳ 已生成 500 条数据...
  ⏳ 已生成 1000 条数据...
  ⏳ 已生成 1500 条数据...
  ⏳ 已生成 2000 条数据...
  ⏳ 已生成 2500 条数据...
  ⏳ 已生成 3000 条数据...
  ⏳ 已生成 3500 条数据...
  ⏳ 已生成 4000 条数据...
  ⏳ 已生成 4500 条数据...
  ⏳ 已生成 5000 条数据...
  ⏳ 已生成 5500 条数据...
  ⏳ 已生成 6000 条数据...

========== 配对完成 ==========
  总 buggy 记录数    : 6978
  成功配对           : 6147
  跳过(无正确版本)   : 831
  跳过(内容完全相同) : 0
  跳过(代码过长)     : 0
✅ 数据集已保存至: deepfix.json  (9359.8 KB)
✅ 数据库连接已关闭


## 4 结果验证与数据预览

In [10]:
def preview_dataset(output_file: str, n: int = 2) -> None:
    """读取保存的 JSON 文件，预览前 n 条数据"""
    with open(output_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    print(f"📊 数据集共 {len(data)} 条，预览前 {n} 条：\n")
    for i, entry in enumerate(data[:n]):
        print(f"{'='*60}")
        print(f"[{i+1}] ID        : {entry['id']}")
        print(f"    user_id   : {entry['user_id']}")
        print(f"    problem_id: {entry['problem_id']}")
        print(f"    error_msg : {entry['error_message'][:80]}...")
        print(f"\n    [buggy_code 前5行]")
        for line in entry['buggy_code'].split('\n')[:5]:
            print(f"      {line}")
        print(f"\n    [fixed_code 前5行]")
        for line in entry['fixed_code'].split('\n')[:5]:
            print(f"      {line}")
        print()


preview_dataset(OUTPUT_FILE, n=2)


📊 数据集共 6147 条，预览前 2 条：

[1] ID        : prog19941
    user_id   : user801
    problem_id: prob90
    error_msg : In function ‘main’:
18:30: error: expected expression before ‘[’ token
         ...

    [buggy_code 前5行]
      #include <stdio.h>
      int main(){
          int k,n;
          scanf("%d%d",&k,&n);
          int a[n];

    [fixed_code 前5行]
      #include <stdio.h>
      int main(){
          int k,n;
          scanf("%d%d",&k,&n);
          int a[n];

[2] ID        : prog00278
    user_id   : user481
    problem_id: prob90
    error_msg : In function ‘main’:
8:9: error: ‘i’ undeclared (first use in this function)
    ...

    [buggy_code 前5行]
      #include <stdio.h>
      int main()
      {
          int k,n,x,a[100];
          scanf("%d", &k);

    [fixed_code 前5行]
      #include <stdio.h>
      int main()
      {
          int k,n,x,m,i,j,a[100];
          scanf("%d", &k);



## 5 随机抽取n条数据（供LLM检查格式）

In [11]:
import json
import random
from pathlib import Path


def sample_dataset(
    json_file: str,
    n: int = 20,
    random_seed: int | None = None,
    verbose: bool = True
) -> list:
    """
    从 DeepFix JSON 数据集中随机抽取 n 条数据并打印显示。

    本函数完全上下文无关，可在 Notebook 的任意 Cell 中独立调用。

    参数:
        json_file    : JSON 文件路径（如 'deepfix_rag_real_pairs.json'）
        n            : 要抽取的数据条数，默认 3
        random_seed  : 随机种子，用于可复现性（默认 None 表示随机）
        verbose      : 是否打印详细信息，默认 True

    返回:
        抽取的数据列表（每条为一个字典）

    异常:
        FileNotFoundError : JSON 文件不存在时抛出
        ValueError        : n 大于数据集总数时抛出
        json.JSONDecodeError : JSON 文件格式不合法时抛出
    """
    import os

    # ---------- 前置校验 ----------
    if not os.path.exists(json_file):
        raise FileNotFoundError(f"JSON 文件不存在: {json_file}")

    # ---------- 加载数据 ----------
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            dataset = json.load(f)
    except json.JSONDecodeError as e:
        raise json.JSONDecodeError(f"JSON 解析失败: {e}", e.doc, e.pos)

    if not isinstance(dataset, list):
        raise ValueError(f"期望 JSON 为列表格式，实际获得: {type(dataset)}")

    total_count = len(dataset)
    if n > total_count:
        raise ValueError(
            f"请求抽取 {n} 条数据，但数据集仅包含 {total_count} 条"
        )

    # ---------- 随机抽样 ----------
    if random_seed is not None:
        random.seed(random_seed)

    sampled_data = random.sample(dataset, n)

    # ---------- 打印输出 ----------
    if verbose:
        print(f"{'='*70}")
        print(f"📊 从 {json_file} 中随机抽取 {n} 条数据（共 {total_count} 条）")
        print(f"{'='*70}\n")

        for idx, entry in enumerate(sampled_data, 1):
            print(f"【样本 {idx}】")
            print(f"  ID          : {entry.get('id', 'N/A')}")
            print(f"  user_id     : {entry.get('user_id', 'N/A')}")
            print(f"  problem_id  : {entry.get('problem_id', 'N/A')}")
            print(f"  bug_type    : {entry.get('bug_type', 'N/A')}")
            print(f"  language    : {entry.get('language', 'N/A')}")

            # ── 错误代码────────────────────────────────
            buggy_code = entry.get('buggy_code', '')
            buggy_lines = buggy_code.split('\n')
            print(f"\n  【buggy_code】")
            for line in buggy_lines:
                print(f"    {line}")

            # ── 正确代码────────────────────────────────
            fixed_code = entry.get('fixed_code', '')
            fixed_lines = fixed_code.split('\n')
            print(f"\n  【fixed_code】")
            for line in fixed_lines:
                print(f"    {line}")

            # ── 错误信息────────────────────────────────────────
            # error_msg = entry.get('error_message', '')
            # if error_msg:
            #     error_preview = error_msg[:80].replace('\n', '\\n')
            #     print(f"\n  【error_message】")
            #     print(f"    {error_preview}{'...' if len(error_msg) > 80 else ''}")
            
            print(f"\n{'-'*70}\n")

    return sampled_data


def sample_dataset_compact(
    json_file: str,
    n: int = 3,
    random_seed: int | None = None
) -> list:
    """
    轻量版本：从 JSON 中随机抽取 n 条数据，仅返回列表，不打印。
    适合在其他函数中作为子函数调用。

    参数:
        json_file   : JSON 文件路径
        n           : 要抽取的数据条数
        random_seed : 随机种子

    返回:
        抽取的数据列表
    """
    return sample_dataset(json_file, n, random_seed, verbose=False)


# ================================================================
# 快速验证：修改下方参数后直接运行本 Cell 即可查看效果
# ================================================================
if __name__ == '__main__' or True:
    _JSON_FILE    = 'deepfix.json'  # ← 修改为实际的 JSON 文件路径
    _SAMPLE_SIZE  = 20                               # ← 修改为想要抽取的条数
    _RANDOM_SEED  = 42                              # ← 固定种子可重现结果，设置 None 则随机

    try:
        samples = sample_dataset(
            _JSON_FILE,
            n=_SAMPLE_SIZE,
            random_seed=_RANDOM_SEED
        )
        print(f"\n✅ 成功抽取 {len(samples)} 条数据")

    except FileNotFoundError as e:
        print(f"❌ {e}")
    except ValueError as e:
        print(f"❌ {e}")
    except json.JSONDecodeError as e:
        print(f"❌ JSON 格式错误: {e}")


📊 从 deepfix.json 中随机抽取 20 条数据（共 6147 条）

【样本 1】
  ID          : prog16773
  user_id     : user720
  problem_id  : prob185
  bug_type    : common_programming_error
  language    : C

  【buggy_code】
    #include <stdio.h>
    int i=0,flag=0;
    void cheak_sum(int a[],int s1,int s2,int cheak,int n){
        
        if(cheak==s1){
            flag=1;
            printf("YES");
        }
        if (flag){
            return;
        }
        i++;
        if(i==n) return 0;
        cheak_sum(a,s1+a[i],s2,cheak)
        cheak_sum(a,s1,s2+a[i],cheak)
    }
    
    
    
    
    
    
    
    
    
    
    
    
    
    int main()
    {
        //Fill this area with your code.
        
        return 0;
    }

  【fixed_code】
    #include <stdio.h>
    int i=0,flag=0;
    void cheak_sum(int a[],int s1,int s2,int cheak,int n){
        if(cheak==s1){
            flag=1;
            printf("YES");
        }
        if (flag){
            return;
        }
        i++;
        if(i==n) retu